In [1]:
%pip install osmnx networkx geopandas shapely folium earthengine-api pandas numpy

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import ee

ee.Authenticate()
ee.Initialize(project='project-0cf410a3-f35f-4911-9c5')

PLACE_NAME = "Tarangire National Park, Tanzania"


Successfully saved authorization token.


In [3]:
from datetime import date
from park_road_network import ParkRoadNetwork
from chirps_provider import CHIRPSProvider
from edge_risk_enricher import EdgeRiskEnricher

import folium

def create_risk_map(lookback_days: int, date: date):

    nodes, edges = ParkRoadNetwork(PLACE_NAME).load()
    rain_provider = CHIRPSProvider()

    pixel_grid = rain_provider.frame_grid(edges.total_bounds)

    rain_pixels, image_collection = rain_provider.get_decayed_precipitation(
        pixel_grid,
        date=date.strftime("%Y-%m-%d"),
        lookback_days=lookback_days
    )

    enricher = EdgeRiskEnricher()
    edges_enriched = enricher.enrich(edges, rain_pixels)

    center_lat, center_lon = nodes["y"].mean(), nodes["x"].mean()
    m = folium.Map(location=[center_lat, center_lon], zoom_start=10)

    SURFACE_RISK_COLORS = {
        (0.0, 0.25): "#2ecc71",
        (0.25, 0.5): "#f39c12",
        (0.5, 0.75): "#e74c3c",
        (0.75, 1.01): "#7b241c",
    }

    def get_risk_color(surface_risk: float) -> str:
        for (low, high), color in SURFACE_RISK_COLORS.items():
            if low <= surface_risk < high:
                return color
        return "#7b241c"

    for _, row in edges_enriched.iterrows():
        coords = [(lat, lon) for lon, lat in row["geometry"].coords]
        road_type = row["road_type"]
        rain_mm = row["rain_mm"]
        travel_time = row["travel_time_m"]

        surface_risk = row["surface_risk"]

        folium.PolyLine(
            locations=coords,
            color=get_risk_color(surface_risk),
            weight=3,
            opacity=0.8,
            popup=(
                f"highway: {road_type}<br>"
                f"travel_time: {travel_time}<br>"
                f"rain_mm(accumualte over x (start - end) days): {rain_mm:.1f}<br>"
                f"surface_risk: {surface_risk:.2f}<br>"
                f"surface_risk: {surface_risk:.2f}"
            ),
        ).add_to(m)

    return m

In [4]:
import ipywidgets as widgets
from IPython.display import display, clear_output
from datetime import date, datetime

date_picker = widgets.DatePicker(
    description="Date:",
    value=date(2024, 4, 30),
)

lookback_days_input = widgets.IntText(
    value=0,
    description="Lookback days:",
    disabled=False,
)

load_button = widgets.Button(
    description="Load Map",
    button_style="primary",
)

output = widgets.Output()


In [5]:
def on_render_clicked(_):
    date = date_picker.value
    lookback_days = lookback_days_input.value

    with output:
        clear_output(wait=True)

        m = create_risk_map(
            date=date,
            lookback_days=lookback_days,
        )
        display(m)


load_button.on_click(on_render_clicked)

In [6]:
display(
    widgets.HBox([date_picker, lookback_days_input, load_button]),
    output,
)

Output()